# Actividad 5 · Cine molecular desde cero
### Dinámica molecular con integración de Verlet (2D, Lennard-Jones)
**Taller:** Física y matemática detrás de los campos de fuerza y la dinámica molecular · XXXIV Semana de la Química, Bogotá 2026

---

**Duración:** ~35 min · **Herramienta:** Google Colab o Jupyter · **Prerrequisito:** Actividades 1 y 2 (Lennard-Jones y descenso de gradiente)

**Objetivos:**
1. Entender qué es una simulación de dinámica molecular: el campo de fuerza **integrado en el tiempo**.
2. Implementar el algoritmo de **Verlet de velocidades** — el estándar de la MD.
3. Usar la **conservación de la energía total** como auditoría de la simulación.
4. Descubrir experimentalmente por qué el paso de integración $dt$ es crítico.

## Concepto clave

La dinámica molecular (MD) aplica el campo de fuerza de las Actividades 1–3 para describir la evolución temporal del sistema. Cada átomo obedece la segunda ley de Newton:

$$\vec{F} = m\,\vec{a}, \qquad \vec{F}_i = -\nabla_i U$$

Dado que el movimiento no puede resolverse analíticamente, se integra **paso a paso**. Uno de los algoritmos más usados es el **Verlet de velocidades**, que en cada paso $dt$ hace:

1. $\vec{x}_{n+1} = \vec{x}_n + \vec{v}_n\,dt + \tfrac{1}{2}\vec{a}_n\,dt^2$  (avanza la posición)
2. Recalcular fuerzas y $\vec{a}_{n+1}$ con las posiciones nuevas
3. $\vec{v}_{n+1} = \vec{v}_n + \tfrac{1}{2}(\vec{a}_n + \vec{a}_{n+1})\,dt$  (actualiza la velocidad)

**Unidades reducidas.** En este ejemplo se usan $\varepsilon=\sigma=m=1$; por tanto, $dt$ se expresa en unidades reducidas de tiempo $\tau=\sigma\sqrt{m/\varepsilon}$. La conversión a tiempo físico requiere los valores de $\sigma$, $\varepsilon$ y la masa. Los pasos habituales de 1–2 fs en simulaciones reales no pueden compararse directamente con este $dt$ sin esa conversión.

> **Criterio de validación:** con un paso suficientemente pequeño, la energía total $E=K+U$ debe conservarse aproximadamente. Se evalúa la desviación máxima durante la trayectoria, no solo la diferencia entre los valores inicial y final.

## Paso 0 · Preparativos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Paso 1 · Fuerzas y energías en 2D (código listo para usar)

Se simulan **dos átomos** en un plano, con interacción de Lennard-Jones (Actividad 1). Se requieren tres funciones:

- `fuerza_lj(p1, p2)`: fuerza vectorial (en 2D) sobre la partícula 1 debida a la 2. Es la $F(r)$ de la Actividad 1 proyectada sobre la dirección que une los átomos.
- `aceleraciones(pos)`: la segunda ley de Newton, $\vec{a} = \vec{F}/m$, para todas las partículas.
- `energias(pos, vel)`: cinética $K = \tfrac{1}{2}m v^2$ y potencial $U$ (Lennard-Jones).

Revisar las funciones e identificar el potencial de la Actividad 1 en el código.

In [ ]:
def fuerza_lj(p1, p2, eps=1.0, sig=1.0):
    """Fuerza (vector 2D) sobre la partícula 1 debida a la partícula 2."""
    rij = p1 - p2
    r2 = np.dot(rij, rij)
    r = np.sqrt(r2)
    # Magnitud de F(r) (Actividad 1) dividida entre r^2, proyectada sobre rij:
    fmag = (24 * eps / r2) * (2 * (sig / r)**12 - (sig / r)**6)
    return fmag * rij

def aceleraciones(pos, m=1.0, eps=1.0, sig=1.0):
    """Segunda ley de Newton: a = F/m para cada partícula."""
    n = len(pos)
    a = np.zeros_like(pos)
    for i in range(n):
        for j in range(n):
            if j != i:
                a[i] += fuerza_lj(pos[i], pos[j], eps, sig) / m
    return a

def energias(pos, vel, m=1.0, eps=1.0, sig=1.0):
    """Energía cinética K y potencial U de la configuración."""
    cinetica = 0.5 * m * np.sum(vel**2)
    potencial = 0.0
    n = len(pos)
    for i in range(n):
        for j in range(i + 1, n):
            r = np.linalg.norm(pos[i] - pos[j])
            potencial += 4 * eps * ((sig / r)**12 - (sig / r)**6)
    return cinetica, potencial

# Prueba rápida: dos átomos a distancia 1.5 sigma
p1, p2 = np.array([0.0, 0.0]), np.array([1.5, 0.0])
print("Fuerza sobre 1:", fuerza_lj(p1, p2), " <- F_x > 0: la atracción jala hacia la 2 (que está a la derecha)")
K, U = energias(np.array([p1, p2]), np.zeros((2, 2)))
print("U =", round(U, 4), " (atractivo, debe ser negativo)")

## Paso 2 · Integración de Verlet: ejercicio

Implementación de las tres actualizaciones del algoritmo de Verlet de velocidades. Las fórmulas aparecen en *Concepto clave*; su traducción a Python es:

| Fórmula | Python |
|---|---|
| $\vec{x}_{n+1} = \vec{x}_n + \vec{v}_n dt + \tfrac{1}{2}\vec{a}_n dt^2$ | `pos + vel*dt + 0.5*ace*dt**2` |
| $\vec{v}_{n+1} = \vec{v}_n + \tfrac{1}{2}(\vec{a}_n + \vec{a}_{n+1}) dt$ | `vel + 0.5*(ace + ace_nueva)*dt` |
| $dt^2$ | `dt**2` |

> Pregunta para después: ¿por qué calculamos primero las posiciones, *luego* las aceleraciones nuevas, y recién entonces las velocidades? (Pista: la velocidad nueva debe usar la fuerza evaluada en el lugar **nuevo**.)

In [ ]:
def paso_verlet(pos, vel, ace, dt, m=1.0, eps=1.0, sig=1.0):
    """Un paso de integración de Verlet de velocidades."""
    # EJERCICIO PENDIENTE 1: nueva posición, usando pos, vel, ace y dt
    # pos_nueva = ...
    # EJERCICIO PENDIENTE 2: aceleraciones con las posiciones NUEVAS
    # ace_nueva = ...
    # EJERCICIO PENDIENTE 3: nueva velocidad, usando ace y ace_nueva
    # vel_nueva = ...
    raise NotImplementedError("Ejercicio pendiente: implementar las tres actualizaciones de Verlet.")
    return pos_nueva, vel_nueva, ace_nueva

## Paso 3 · La "película": dos átomos se encuentran

Escena inicial: dos átomos a distancia $4\,\sigma$. El átomo 1 se mueve hacia el 2 con velocidad $v_0 = 3$. ¿Qué comportamiento se espera? (Sugerencia: consultar la gráfica de $U(r)$ de la Actividad 1.) La figura y la animación muestran los primeros 600 pasos; la trayectoria completa se conserva para el análisis energético. Con $dt=0.005$, $v_0=3$ y 4000 pasos, la desviación máxima relativa esperada es aproximadamente $1.9\times10^{-3}$; la diferencia final es menor y no refleja el máximo intermedio.

Ejecutar la simulación y analizar:
1. **El intercambio energía cinética ↔ potencial**: cuando se acercan, $U$ sube y $K$ baja; cuando se alejan, al revés.
2. **La energía total debe permanecer aproximadamente constante.** La desviación máxima cuantifica la precisión de la integración.

In [ ]:
def simular(dt=0.005, pasos=4000, v0=3.0):
    pos = np.array([[3.0, 5.0], [7.0, 5.0]])    # dos átomos a distancia 4*sigma
    vel = np.array([[v0, 0.0], [0.0, 0.0]])     # el átomo 1 se acerca al 2
    ace = aceleraciones(pos)
    traj = np.zeros((pasos + 1, 2, 2)); traj[0] = pos
    EK = np.zeros(pasos + 1); EP = np.zeros(pasos + 1)
    EK[0], EP[0] = energias(pos, vel)
    for t in range(pasos):
        pos, vel, ace = paso_verlet(pos, vel, ace, dt)
        traj[t + 1] = pos
        EK[t + 1], EP[t + 1] = energias(pos, vel)
    return traj, EK, EP

traj, EK, EP = simular(dt=0.005, pasos=4000, v0=3.0)
ET = EK + EP

print("Auditoría de energía:")
print(f"  E inicial = {ET[0]:.6f}")
print(f"  E final   = {ET[-1]:.6f}")
deriva_max = np.max(np.abs(ET - ET[0])) / abs(ET[0])
deriva_final = abs(ET[-1] - ET[0]) / abs(ET[0])
print(f"  desviación máxima relativa = {deriva_max:.2e}")
print(f"  desviación final relativa  = {deriva_final:.2e}")
print(f"  distancia mínima alcanzada: {np.min(np.linalg.norm(traj[:,0]-traj[:,1], axis=1)):.3f} sigma")

# --- Figura 1: energías vs tiempo ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
t = np.arange(len(ET)) * 0.005
ax1.plot(t, EK, label='cinética K', color='navy')
ax1.plot(t, EP, label='potencial U', color='darkred')
ax1.plot(t, ET, label='TOTAL (aproximada)', color='black', lw=2.5)
ax1.set_xlabel('tiempo'); ax1.set_ylabel('energía')
ax1.set_title('Intercambio K <-> U; desviación máxima de energía reportada')
ax1.legend(); ax1.grid(alpha=0.3)

# --- Figura 2: fotogramas de la película (SU PROPIA figura) ---
frames = [0, 100, 150, 200, 250, 300, 400, 600]
fig2, axes2 = plt.subplots(2, 4, figsize=(13, 5.5))
for ax, frame in zip(axes2.flat, frames):
    ax.plot(traj[frame, 0, 0], traj[frame, 0, 1], 'o', color='navy', ms=12)
    ax.plot(traj[frame, 1, 0], traj[frame, 1, 1], 'o', color='darkred', ms=12)
    ax.plot(traj[:frame+1, 0, 0], traj[:frame+1, 0, 1], '-', color='navy', alpha=0.4)
    ax.plot(traj[:frame+1, 1, 0], traj[:frame+1, 1, 1], '-', color='darkred', alpha=0.4)
    ax.set_title(f'frame {frame}', fontsize=9)
    ax.set_xlim(0, 14); ax.set_ylim(3, 7)
    ax.set_xticks([]); ax.set_yticks([])
fig2.suptitle('La película molecular: acercamiento, "choque" y rebote', y=0.98)
plt.tight_layout()
plt.show()

## Experimento · ¿Qué paso de integración $dt$ es demasiado grande?

Todo integrador numérico tiene un límite de estabilidad. Al aumentar $dt$, la energía total puede dejar de conservarse y la trayectoria puede apartarse de la solución física.

Ejecutar el barrido y determinar el límite de estabilidad. Criterio de explosión: la energía total se desvía más de un 50 % de su valor inicial en algún momento de la trayectoria.

In [ ]:
print(f"{'dt':>6} | {'deriva máx. de E':>18} | veredicto")
print("-" * 45)
for dt in [0.005, 0.01, 0.02, 0.04, 0.06]:
    traj, EK, EP = simular(dt=dt, pasos=3000, v0=3.0)
    ET = EK + EP
    deriva_max = np.max(np.abs(ET - ET[0])) / abs(ET[0])
    veredicto = "estable" if deriva_max < 0.01 else ("revisar" if deriva_max < 0.5 else "inestable")
    print(f"{dt:>6} | {deriva_max:>18.2e} | {veredicto}")

# Gráfica: energía total para cada dt
fig, axes = plt.subplots(1, 5, figsize=(17, 3.4), sharey=True)
for ax, dt in zip(axes, [0.005, 0.01, 0.02, 0.04, 0.06]):
    traj, EK, EP = simular(dt=dt, pasos=3000, v0=3.0)
    ax.plot(EK + EP, lw=1)
    ax.set_title(f"dt = {dt}", fontsize=10)
    ax.set_xlabel('paso'); ax.grid(alpha=0.3)
axes[0].set_ylabel('energía total')
plt.suptitle('La energía total debe ser horizontal: ¿desde qué dt deja de serlo?', y=1.05)
plt.tight_layout()
plt.show()

## (Opcional) La película en movimiento

En Google Colab, esta celda genera un video de la trayectoria. En Jupyter clásico puede requerirse ffmpeg; si no está disponible, se pueden consultar los fotogramas anteriores.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

traj, EK, EP = simular(dt=0.005, pasos=600, v0=3.0)
fig, ax = plt.subplots(figsize=(7, 3))
ax.set_xlim(0, 14); ax.set_ylim(3.5, 6.5)
ax.set_xticks([]); ax.set_yticks([])
punto1, = ax.plot([], [], 'o', color='navy', ms=14)
punto2, = ax.plot([], [], 'o', color='darkred', ms=14)
cola1, = ax.plot([], [], '-', color='navy', alpha=0.4)
cola2, = ax.plot([], [], '-', color='darkred', alpha=0.4)

def cuadro(k):
    punto1.set_data([traj[k,0,0]], [traj[k,0,1]])
    punto2.set_data([traj[k,1,0]], [traj[k,1,1]])
    cola1.set_data(traj[:k+1,0,0], traj[:k+1,0,1])
    cola2.set_data(traj[:k+1,1,0], traj[:k+1,1,1])
    return punto1, punto2, cola1, cola2

anim = animation.FuncAnimation(fig, cuadro, frames=range(0, 601, 3), interval=40, blit=True)
plt.close(fig)
HTML(anim.to_html5_video())

## Síntesis · Elementos desarrollados

| Ingrediente | De dónde viene |
|---|---|
| $U(r)$ de Lennard-Jones | Actividad 1 (el potencial) |
| $\vec{F} = -\nabla U$ | Actividad 1 (la derivada) |
| Descenso controlado por fuerzas | Actividad 2 (misma idea, ahora con inercia: la velocidad "carga" el movimiento) |
| Conservación aproximada de $E$ | La desviación máxima permite auditar la integración |
| Paso temporal | $dt$ está en unidades reducidas; 1–2 fs es una escala habitual en sistemas reales y requiere conversión física |

> La MD consiste en integrar el campo de fuerzas **en el tiempo**. Los conceptos de dinámica molecular y RMSD se interpretan a partir de los cálculos realizados en cada paso.

## Cierre · Conexión con el *docking molecular*

1. **¿Para qué sirve la MD en el flujo del curso?** El docking propone poses estáticas; la MD permite evaluar su evolución en el tiempo y observar si el complejo proteína-ligando se mantiene unido bajo las condiciones simuladas.

2. **Escala real.** El ejemplo simula dos átomos en unidades reducidas. Una MD de un sistema físico puede incluir decenas de miles de átomos y millones de pasos; la escala de 1–2 fs requiere convertir las unidades con los parámetros físicos del sistema.

3. **¿Dónde estamos en el mapa del taller?**
   - Actividades 1–3: potencial, paisaje energético y campo de fuerza.
   - Actividad 4 (en el aula): mapa de campos de fuerza.
   - **Actividad 5:** dinámica molecular desde cero y evaluación temporal de poses.
   - Actividad 6: analizar una trayectoria real → ¿estable o no estable? (RMSD).

> **Pregunta guía del curso:** una hipótesis de unión honesta no es una pose bonita, sino una que **sobrevive el tiempo**. La MD permite evaluar esta estabilidad.